# [skip-ci]

# Cell-Type Enrichment with ViralScan

This vignette demonstrates the `--cell-types` workflow and the
`viralscan.enrichment` Python API.

## What we will do

1. Load a ViralScan AnnData result (or create a synthetic one for demonstration).
2. Prepare a `barcode,cell_type` CSV mapping.
3. Call `cell_type_enrichment()` to compute per-virus Fisher exact tests.
4. Write the results table with `write_cell_type_enrichment()`.
5. Visualise enrichment p-values with a bar chart.

## Prerequisites

- ViralScan installed (`pip install ViralScan`).
- `anndata`, `pandas`, `scipy`, `matplotlib` available.
- A completed ViralScan run (e.g. from `basic_usage.ipynb`), **or** the
  synthetic data path taken in Step 1 below.

## Step 1 — Load AnnData (real or synthetic)

If you ran `basic_usage.ipynb` first, set `USE_REAL = True` and point
`h5ad_path` to your output directory.  Otherwise we build a small synthetic
AnnData so this notebook is self-contained.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import anndata as ad
import scipy.sparse as sp

USE_REAL = False  # set True to load from a previous viralscan run
h5ad_path = Path("viralscan_output") / "results" / "adata_multimap.h5ad"

if USE_REAL and h5ad_path.exists():
    import scanpy as sc
    adata = sc.read_h5ad(h5ad_path)
    print(f"Loaded real AnnData: {adata}")
else:
    # Synthetic AnnData: 200 cells × 10 genes (8 host + 2 viral)
    rng = np.random.default_rng(42)
    n_cells, n_genes = 200, 10
    barcodes = [f"CELL{i:04d}-1" for i in range(n_cells)]
    gene_names = [
        "GAPDH", "ACTB", "CD3D", "CD14", "MS4A1", "FCGR3A", "NKG7", "LYZ",
        "INFLUENZA_A_PB2", "INFLUENZA_A_NS1",
    ]
    counts = rng.negative_binomial(5, 0.5, size=(n_cells, n_genes)).astype(np.float32)
    # Make viral genes sparse (only ~10 % of cells have viral reads)
    viral_mask = rng.random((n_cells, 2)) > 0.9
    counts[:, 8:] = counts[:, 8:] * viral_mask
    adata = ad.AnnData(
        X=sp.csr_matrix(counts),
        obs=pd.DataFrame(index=barcodes),
        var=pd.DataFrame(index=gene_names),
    )
    print(f"Synthetic AnnData: {adata}")

## Step 2 — Prepare a barcode → cell-type CSV

The `--cell-types` flag and the Python API both expect a two-column CSV with
headers `barcode` and `cell_type`.  Each row maps one cell barcode to its
annotated cell type.

In a real analysis these labels would come from a Seurat or scanpy clustering
run.  Here we assign mock labels.

In [ ]:
import tempfile

# Assign cell types round-robin: T cell, B cell, Monocyte, NK cell
cell_type_labels = ["T cell", "B cell", "Monocyte", "NK cell"]
barcodes_list = adata.obs_names.tolist()
assigned = [cell_type_labels[i % len(cell_type_labels)] for i in range(len(barcodes_list))]

cell_types_df = pd.DataFrame({"barcode": barcodes_list, "cell_type": assigned})

# Write to a temporary CSV (in practice, point to your annotation file)
tmp_dir = Path(tempfile.mkdtemp())
cell_types_csv = tmp_dir / "cell_types.csv"
cell_types_df.to_csv(cell_types_csv, index=False)
print(f"Cell-types CSV written to: {cell_types_csv}")
cell_types_df.head(8)

## Step 3 — Run `cell_type_enrichment()`

We import directly from `viralscan.enrichment` — no Snakemake environment
needed.

In [ ]:
from viralscan.enrichment import cell_type_enrichment, write_cell_type_enrichment

# group_by_virus maps each virus name to its gene/feature names in adata.var
group_by_virus = {
    "Influenza A": ["INFLUENZA_A_PB2", "INFLUENZA_A_NS1"],
}

cfg = {"cell_types": str(cell_types_csv)}

enrichment_df = cell_type_enrichment(adata, group_by_virus, cfg)
print(f"Enrichment result shape: {enrichment_df.shape}")
enrichment_df

## Step 4 — Write the enrichment TSV

In [ ]:
output_dir = tmp_dir / "viralscan_enrichment_demo"
output_dir.mkdir(exist_ok=True)

out_path = write_cell_type_enrichment(enrichment_df, str(output_dir))

if out_path:
    written = Path(out_path)
    print(f"TSV written: {written}")
    print(f"File size: {written.stat().st_size} bytes")
    # Verify it round-trips
    reloaded = pd.read_csv(written, sep="\t")
    print(f"Reloaded rows: {len(reloaded)}")
else:
    print("Nothing written (empty enrichment result).")

## Step 5 — Bar chart of enrichment p-values

In [ ]:
import matplotlib.pyplot as plt

if enrichment_df.empty:
    print("No enrichment data to plot.")
else:
    fig, ax = plt.subplots(figsize=(7, 4))

    colors = ["tab:red" if p < 0.05 else "tab:gray" for p in enrichment_df["padj"]]
    bars = ax.bar(enrichment_df["cell_type"], -np.log10(enrichment_df["padj"] + 1e-10), color=colors)

    ax.axhline(-np.log10(0.05), color="red", linestyle="--", linewidth=0.8, label="padj = 0.05")
    ax.set_xlabel("Cell type")
    ax.set_ylabel("-log10(padj)")
    ax.set_title("Cell-type enrichment — Influenza A")
    ax.legend()
    plt.tight_layout()
    plt.show()

## Summary and interpretation

The enrichment table contains one row per virus × cell-type combination with:

| Column | Meaning |
|--------|---------|
| `n_infected` | Infected cells in this cell type |
| `n_total` | All labeled cells of this type |
| `pct` | % of cells in this type with ≥1 viral UMI |
| `OR` | Fisher exact odds ratio (one-sided, "greater") |
| `pvalue` | Raw Fisher p-value |
| `padj` | Benjamini-Hochberg adjusted p-value |

Cell types with `padj < 0.05` (red bars) are over-represented among infected
cells relative to all other labeled cells.

**In a real dataset** you would:
1. Replace the synthetic `group_by_virus` dict with gene names that actually
   appear in your ViralScan output (`adata.var_names`).
2. Supply a real annotation CSV from your cell-type calling workflow.
3. Pass `--cell-types your_annotation.csv` to `viralscan` directly to generate
   the TSV automatically as part of the pipeline run.